
### Restart run

In [1]:

from __future__ import division

import numpy as np
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_sun, Angle, Longitude
from sunpy.coordinates import frames, sun
import star_chart_spherical_projection as scsp
from scipy.stats import gaussian_kde
import random
import pandas as pd
import csv
import argparse
import os
import errno

# supress dubious year warnings 
import warnings
warnings.simplefilter('ignore', UserWarning)

In [2]:
def dS_offset(year):
    '''
    Introduce offset number of days to align Stellarium and Astropy JD. 
    Tested for 1600 to 1100 BCE. 
    '''
    if int(year) > 1499:
        dS_off = -14 
    elif int(year) < 1201:    
        dS_off = -11
    else:
        dS_off = 1 - (int(year) / 100.0)
    return dS_off

In [3]:
# inputs
year = '1300'
month = '01'
matchStellariumJD = True

In [99]:
def readMockCoords(path):
    '''
    Read mock coordinates from an existing random starfield. 
    '''
    # initalize empty key lists
    df = pd.read_csv(path, sep=',')
    RA_list = list(df['RA'])
    Dec_list = list(df['Dec'])
    obj_list = SkyCoord(RA_list * u.hour, Dec_list * u.deg)
    return(obj_list)  

In [103]:
direct = os.getcwd() # current working directory
direct = direct + '/StarLists/RandSky/ICs/' # directory where the .txt files go

filename = 'mockdata_518_1300BC-Mar-18-2024_1059.txt' 
filename2 = 'star_data_' + filename[20:-3]+ 'csv'


df2 = pd.read_csv(direct + filename2, sep=',')

In [110]:
(nstar, trash) = df2.shape

In [111]:
nstar

518

In [112]:
(trash, nhead) = df.shape

In [113]:
nhead

1040

In [114]:
2 * nstar + 4

1040

In [104]:
direct = os.getcwd() # current working directory
direct = direct + '/StarLists/RandSky/' # directory where the .txt files go

filename = 'mockdata_518_1300BC-Mar-18-2024_1059.txt' 

df = pd.read_csv(direct + filename, sep='|')


In [105]:
df.shape

(46558, 1040)

In [91]:
hd_list = list(df)

['Julian Date',
 'Local Date and Time',
 'Sun Azimuth',
 'Sun Altitude',
 'R0000 Azimuth',
 'R0000 Altitude',
 'R0001 Azimuth',
 'R0001 Altitude',
 'R0002 Azimuth',
 'R0002 Altitude',
 'R0003 Azimuth',
 'R0003 Altitude',
 'R0004 Azimuth',
 'R0004 Altitude',
 'R0005 Azimuth',
 'R0005 Altitude',
 'R0006 Azimuth',
 'R0006 Altitude',
 'R0007 Azimuth',
 'R0007 Altitude',
 'R0008 Azimuth',
 'R0008 Altitude',
 'R0009 Azimuth',
 'R0009 Altitude',
 'R0010 Azimuth',
 'R0010 Altitude',
 'R0011 Azimuth',
 'R0011 Altitude',
 'R0012 Azimuth',
 'R0012 Altitude',
 'R0013 Azimuth',
 'R0013 Altitude',
 'R0014 Azimuth',
 'R0014 Altitude',
 'R0015 Azimuth',
 'R0015 Altitude',
 'R0016 Azimuth',
 'R0016 Altitude',
 'R0017 Azimuth',
 'R0017 Altitude',
 'R0018 Azimuth',
 'R0018 Altitude',
 'R0019 Azimuth',
 'R0019 Altitude',
 'R0020 Azimuth',
 'R0020 Altitude',
 'R0021 Azimuth',
 'R0021 Altitude',
 'R0022 Azimuth',
 'R0022 Altitude',
 'R0023 Azimuth',
 'R0023 Altitude',
 'R0024 Azimuth',
 'R0024 Altitude',
 '

In [5]:
lastline = df.tail(1)

index = df.tail(1).index.item()
dat = df.iloc[-1]["Local Date and Time"]

jdinit = df.iloc[0]["Julian Date"]
jdsave = df.iloc[-1]["Julian Date"]

savhour = int(dat[13:15])
savmin = int(dat[16:18])//4

In [118]:
int(dat[1:6])

1300

In [115]:
dat

'-01300-05-10T07:48:00.000'

In [8]:

# # Set Location on Earth (currently hardcoded to Luxor, Egypt)
Luxor = EarthLocation(lat=25.6989*u.deg, lon=32.6421*u.deg, height=89*u.m) # data matched to Stellarium

# # Times and dates
# hour and minute steps
dhour = 0.04166666674427688 #iterate every hour
d4min = 0.00277777784503996 # iterate every 4 minutes


# introduce offset of a number of days to get JD in line with Stellarium:
dS = 0
if matchStellariumJD:
    dS = dS_offset(year)



In [9]:
# star time to jd with offset for local sidereal time
start = (Time('-0' + year + '-' + month + '-01T00:00:00.000', scale="local", location = Luxor).jd) - (Luxor.lon.deg/15.0) * dhour + dS
print(start)


# days = start + 1 * np.arange(int(jdsave - jdinit), 365) # iterate for a year 
hours = dhour * np.arange(savhour, 24)
minutes = d4min * np.arange(savmin + 1, 15)

1246232.4093274998


In [11]:
savmin

12

In [44]:
endtime - start 

364.9972222249489

In [46]:
jdsave

1246361.7343275012

In [55]:
(endtime - jdsave) / d4min

84841.99794609862

In [78]:
numiter = round((endtime - jdsave) / d4min)

In [58]:
range(0, numiter)

range(0, 84842)

In [64]:
dat

'-01300-05-10T07:48:00.000'

In [80]:
for i in range (numiter-5, numiter):
    temptime = jdsave + d4min * i + d4min
    print(str(Time(temptime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))

-01300-12-31T23:40:00.493
-01300-12-31T23:44:00.493
-01300-12-31T23:48:00.493
-01300-12-31T23:52:00.493
-01300-12-31T23:56:00.493


In [77]:
days = start + 1 * np.arange(0, 365) # iterate for a year 
hours = dhour * np.arange(0, 24)
minutes = d4min * np.arange(0, 15)

endtime = days[-1] + hours[-1] + minutes[-1] 
print(str(Time(endtime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))

-01300-12-31T23:56:00.000


In [39]:
endtime = start + 364 + hours[-1] + minutes[-1]
Time(testtime - dS + (Luxor.lon.deg/15.0) * dhour, format='jd').fits

'-01300-12-31T23:56:00.000'

In [29]:
Time(dat, format='fits').jd + d4min

1246373.8277777778

In [26]:
# finish hour
day = start + int(jdsave - jdinit)
hour = hours[0]
for mins in minutes:
    temptime = day + hour + mins
    print(str(Time(temptime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))

#reset minutes
minutes = d4min * np.arange(0, 15)

# finish day
for hour in hours:
    for mins in minutes:
        temptime = day + hour + mins
        print(str(Time(temptime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))

# reset hours and days
hours = dhour * np.arange(0, 24)
days = start + 1 * np.arange(int(jdsave - jdinit) + 1, 365) # iterate for a year 

# # finish year
# i = 0
# for day in days:
#     for hour in hours:
#         for mins in minutes:
#             temptime = day + hour + mins
#             i+=1
#             print(str(Time(temptime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))
#             if i > 1000:
#                 break
#         else:
#             continue
#         break
#     else:
#         continue
#     break

-01300-05-10T00:00:00.000
-01300-05-10T00:04:00.000
-01300-05-10T00:08:00.000
-01300-05-10T00:12:00.000
-01300-05-10T00:16:00.000
-01300-05-10T00:20:00.000
-01300-05-10T00:24:00.000
-01300-05-10T00:28:00.000
-01300-05-10T00:32:00.000
-01300-05-10T00:36:00.000
-01300-05-10T00:40:00.000
-01300-05-10T00:44:00.000
-01300-05-10T00:48:00.000
-01300-05-10T00:52:00.000
-01300-05-10T00:56:00.000
-01300-05-10T00:00:00.000
-01300-05-10T00:04:00.000
-01300-05-10T00:08:00.000
-01300-05-10T00:12:00.000
-01300-05-10T00:16:00.000
-01300-05-10T00:20:00.000
-01300-05-10T00:24:00.000
-01300-05-10T00:28:00.000
-01300-05-10T00:32:00.000
-01300-05-10T00:36:00.000
-01300-05-10T00:40:00.000
-01300-05-10T00:44:00.000
-01300-05-10T00:48:00.000
-01300-05-10T00:52:00.000
-01300-05-10T00:56:00.000
-01300-05-10T01:00:00.000
-01300-05-10T01:04:00.000
-01300-05-10T01:08:00.000
-01300-05-10T01:12:00.000
-01300-05-10T01:16:00.000
-01300-05-10T01:20:00.000
-01300-05-10T01:24:00.000
-01300-05-10T01:28:00.000
-01300-05-10

In [13]:
# i = 0

# for day in days:
#     for hour in hours:
#         for mins in minutes:
#             temptime = day + hour + mins
#             i+=1
#             print(str(Time(temptime - dS + (Luxor.lon.deg/15.0) * dhour, format = 'jd').fits))
#             if i > 10:
#                 break
#         else:
#             continue
#         break
#     else:
#         continue
#     break

In [25]:
path = 'blabla'
abspath = os.path.join(os.getcwd(), path)

if not os.path.exists(abspath):
    error = "ERROR: file does not exist at " + abspath
    raise Exception(error)

Exception: ERROR: file does not exist at /Users/lunazagor/Code/GitHub/decanOpy/blabla

In [23]:

abspath = os.path.join(os.getcwd(), path)

In [24]:
"ERROR: file does not exist at " + abspath

'ERROR: file does not exist at /Users/lunazagor/Code/GitHub/decanOpy/blabla'